# 04 - CRE Investment LGD Model

**Office, Retail, Industrial, Mixed-Use - Australian Bank-Style Portfolio Framework**

This notebook demonstrates a practical CRE Investment LGD framework:

| Step | Description |
|---|---|
| 1 | Generate synthetic CRE investment default dataset |
| 2 | Segment by asset class and key risk drivers |
| 3 | Build base LGD, downturn LGD, and weighted LGD outputs |
| 4 | Differentiate refinance vs forced sale recovery path |
| 5 | Run sensitivity analysis on cap-rate, vacancy, and tenant concentration |


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(46)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")



## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** transparent proxy drivers are permitted where observed workout tapes are unavailable.
- **Cure / resolution treatment:** CRE investment uses refinance vs forced-sale channel logic; no mortgage-style borrower cure model is forced into this module.
- **Calibration status (standard policy):** this notebook is a transparent proxy baseline and is **not** internally calibrated to real workout tapes.
- **Output standard:** report `lgd_base`, `lgd_downturn`, `lgd_final`, EAD-weighted outputs, and base recovery-time metric (`time_to_recovery_months_base`).


## Objective
Build an interview-ready CRE investment LGD module with refinance-vs-forced-sale channel logic and weighted outputs.

## Drivers
- LVR and DSCR
- WALE and vacancy
- Tenant concentration
- Cap-rate expansion and resolution path

## Logic
- Base/downturn/final LGD with explicit recovery-time metric and weighted aggregation

## Downturn
- Macro-linked stress via cap-rate expansion, weaker occupancy, and slower resolution

## Validation
- Weighted segment/path outputs, scenario sensitivity checks, and validation flags

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## Why CRE Investment Differs from Mortgage and Development

CRE investment LGD is primarily driven by **income durability and refinancing risk** for stabilised assets:

- Versus mortgage: CRE is less about household cure/LMI and more about DSCR, WALE, vacancy, and tenant concentration.
- Versus development: CRE is less about completion/cost-to-complete execution and more about cap-rate repricing and exit liquidity.
- Recovery path is often a choice between refinance and forced sale under stressed cap rates.


## 1. Generate Synthetic CRE Investment Dataset


In [ ]:
n_loans = 280
rng = np.random.default_rng(46)

segments = ['office', 'retail', 'industrial', 'mixed-use']
segment_weights = [0.30, 0.23, 0.32, 0.15]
segment = rng.choice(segments, size=n_loans, p=segment_weights)

segment_params = {
    'office': {'value_low': 20e6, 'value_high': 180e6, 'lvr_mean': 0.64, 'dscr_mean': 1.55, 'wale_mean': 4.8, 'vac_mean': 0.08, 'tenant_conc_mean': 0.34, 'cap_rate': 0.055},
    'retail': {'value_low': 15e6, 'value_high': 140e6, 'lvr_mean': 0.66, 'dscr_mean': 1.45, 'wale_mean': 4.0, 'vac_mean': 0.10, 'tenant_conc_mean': 0.38, 'cap_rate': 0.060},
    'industrial': {'value_low': 18e6, 'value_high': 160e6, 'lvr_mean': 0.62, 'dscr_mean': 1.70, 'wale_mean': 5.3, 'vac_mean': 0.06, 'tenant_conc_mean': 0.30, 'cap_rate': 0.053},
    'mixed-use': {'value_low': 22e6, 'value_high': 190e6, 'lvr_mean': 0.67, 'dscr_mean': 1.40, 'wale_mean': 3.6, 'vac_mean': 0.11, 'tenant_conc_mean': 0.42, 'cap_rate': 0.062},
}

rows = []
for i, seg in enumerate(segment, start=1):
    p = segment_params[seg]
    property_value = rng.uniform(p['value_low'], p['value_high'])
    lvr = float(np.clip(rng.normal(p['lvr_mean'], 0.08), 0.40, 0.90))
    ead = property_value * lvr * rng.uniform(0.98, 1.03)
    dscr = float(np.clip(rng.normal(p['dscr_mean'], 0.30), 0.75, 3.20))
    icr = float(np.clip(rng.normal(dscr + 0.05, 0.20), 0.80, 3.50))
    wale = float(np.clip(rng.normal(p['wale_mean'], 1.4), 0.5, 9.5))
    vacancy_rate = float(np.clip(rng.normal(p['vac_mean'], 0.05), 0.01, 0.35))
    tenant_concentration = float(np.clip(rng.normal(p['tenant_conc_mean'], 0.10), 0.12, 0.90))

    cap_rate_base = float(np.clip(rng.normal(p['cap_rate'], 0.005), 0.045, 0.085))
    cap_rate_expansion = float(np.clip(rng.normal(0.018, 0.010), 0.000, 0.080))

    contract_rate_proxy = float(np.clip(rng.normal(0.062, 0.010), 0.040, 0.095))
    cost_of_funds_proxy = float(np.clip(rng.normal(0.051, 0.008), 0.035, 0.080))
    discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)

    refinance_score = (
        0.90 * (dscr - 1.20)
        + 0.35 * (wale - 3.0) / 3.0
        - 2.20 * (lvr - 0.60)
        - 1.10 * vacancy_rate
        - 0.90 * tenant_concentration
        + rng.normal(0, 0.20)
    )
    refinance_prob = 1.0 / (1.0 + np.exp(-refinance_score))
    refinance_flag = int(rng.uniform() < refinance_prob)
    resolution_path = 'refinance' if refinance_flag == 1 else 'forced_sale'

    rows.append({
        'loan_id': f'CRE{i:04d}',
        'segment': seg,
        'property_value': property_value,
        'ead': ead,
        'lvr_at_default': lvr,
        'dscr': dscr,
        'icr': icr,
        'wale_years': wale,
        'vacancy_rate': vacancy_rate,
        'tenant_concentration': tenant_concentration,
        'cap_rate_base': cap_rate_base,
        'cap_rate_expansion': cap_rate_expansion,
        'contract_rate_proxy': contract_rate_proxy,
        'cost_of_funds_proxy': cost_of_funds_proxy,
        'discount_rate': discount_rate,
        'refinance_prob': refinance_prob,
        'refinance_flag': refinance_flag,
        'forced_sale_flag': 1 - refinance_flag,
        'resolution_path': resolution_path,
    })

cre = pd.DataFrame(rows)

print('CRE loans:', cre.shape)
display(cre['segment'].value_counts())
cre.head()



## 2. Build Base and Downturn LGD


In [ ]:
# Base LGD for CRE investment (auditable additive proxy)
cre['lgd_base'] = (
    0.18
    + 0.34 * (cre['lvr_at_default'] - 0.55).clip(lower=0, upper=0.35)
    + 0.20 * ((1.40 - cre['dscr']) / 1.40).clip(lower=0, upper=1.00)
    + 0.08 * ((4.0 - cre['wale_years']) / 4.0).clip(lower=0, upper=1.00)
    + 0.16 * cre['vacancy_rate'].clip(0, 0.35)
    + 0.15 * cre['tenant_concentration'].clip(0.10, 0.90)
    + 0.10 * cre['forced_sale_flag']
    - 0.05 * cre['refinance_flag']
).clip(0.05, 0.95)

# Downturn stress linked to cap-rate expansion, vacancy stress, and forced-sale channel
cre['downturn_scalar'] = (
    1.00
    + 2.10 * cre['cap_rate_expansion'].clip(0.0, 0.08)
    + 0.45 * cre['vacancy_rate'].clip(0.0, 0.35)
    + 0.12 * cre['forced_sale_flag']
).clip(1.00, 1.70)

cre['lgd_downturn'] = (cre['lgd_base'] * cre['downturn_scalar']).clip(0.0, 0.99)

# Recovery-time metric used by cross-product comparison:
# expected months from default to economic resolution (refinance settlement or forced-sale completion).
cre['time_to_recovery_months_base'] = (
    8.0
    + 10.0 * cre['forced_sale_flag']
    + 8.0 * cre['vacancy_rate'].clip(0.0, 0.35)
    + 6.0 * (1.0 - cre['refinance_prob'])
    + 50.0 * cre['cap_rate_expansion'].clip(0.0, 0.08)
).clip(4.0, 48.0)

cre['time_to_recovery_months_downturn'] = (
    cre['time_to_recovery_months_base']
    * (1.12 + 0.25 * cre['cap_rate_expansion'].clip(0.0, 0.08) + 0.10 * cre['forced_sale_flag'])
).clip(6.0, 60.0)

# Explicit final overlay (portfolio-demo conservatism) so cross-product comparison is apples-to-apples.
cre['final_overlay_addon'] = (
    0.010
    + 0.020 * cre['forced_sale_flag']
    + 0.012 * cre['vacancy_rate'].clip(0.0, 0.35)
    + 0.012 * (1.0 - cre['refinance_prob'])
).clip(0.010, 0.070)

cre['lgd_final'] = (cre['lgd_downturn'] + cre['final_overlay_addon']).clip(0.0, 0.99)

ewa_base = exposure_weighted_average(cre, 'lgd_base', 'ead')
ewa_downturn = exposure_weighted_average(cre, 'lgd_downturn', 'ead')
ewa_final = exposure_weighted_average(cre, 'lgd_final', 'ead')

print(f'EAD-weighted base LGD:      {ewa_base:.2%}')
print(f'EAD-weighted downturn LGD:  {ewa_downturn:.2%}')
print(f'EAD-weighted final LGD:     {ewa_final:.2%}')

display(cre[['lgd_base', 'downturn_scalar', 'lgd_downturn', 'final_overlay_addon', 'lgd_final']].describe().round(4))



## 3. Segment-Level Weighted LGD Outputs


In [ ]:
rows = []
for seg, grp in cre.groupby('segment', observed=True):
    rows.append({
        'segment': seg,
        'loan_count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd_base': exposure_weighted_average(grp, 'lgd_base', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(grp, 'lgd_downturn', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(grp, 'lgd_final', 'ead'),
        'mean_lvr': grp['lvr_at_default'].mean(),
        'mean_dscr': grp['dscr'].mean(),
        'mean_wale_years': grp['wale_years'].mean(),
        'mean_vacancy_rate': grp['vacancy_rate'].mean(),
        'mean_tenant_concentration': grp['tenant_concentration'].mean(),
        'mean_time_to_recovery_months_base': grp['time_to_recovery_months_base'].mean(),
    })

segment_summary = pd.DataFrame(rows).sort_values('ead_weighted_lgd_final', ascending=False).reset_index(drop=True)
display(segment_summary.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = segment_summary.set_index('segment')[
    ['ead_weighted_lgd_base', 'ead_weighted_lgd_downturn', 'ead_weighted_lgd_final']
]
plot_df.plot(kind='bar', ax=ax, color=['#4c72b0', '#dd8452', '#c44e52'])
ax.set_title('CRE Segment LGD - EAD Weighted Base vs Downturn vs Final')
ax.set_ylabel('LGD')
ax.legend(['Base LGD', 'Downturn LGD', 'Final LGD'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'cre_segment_weighted_lgd.png'), dpi=140, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'cre_investment_weighted_lgd_comparison.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
    print('Plot display skipped (set LGD_NOTEBOOK_SHOW_PLOTS=1 to show).')



## 4. Refinance vs Forced Sale Channel


In [ ]:
path_summary = (
    cre.groupby('resolution_path', observed=True)
    .apply(
        lambda g: pd.Series({
            'loan_count': len(g),
            'total_ead': g['ead'].sum(),
            'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base', 'ead'),
            'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead'),
            'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final', 'ead'),
            'mean_cap_rate_expansion': g['cap_rate_expansion'].mean(),
            'mean_vacancy_rate': g['vacancy_rate'].mean(),
            'mean_time_to_recovery_months_base': g['time_to_recovery_months_base'].mean(),
        }),
        include_groups=False,
    )
    .reset_index()
)
display(path_summary.round(4))



## 5. Sensitivity Analysis

Sensitivity is run on three CRE-specific stress levers:
- cap-rate expansion
- vacancy shock
- tenant concentration shock


In [ ]:
def run_cre_sensitivity(df, cap_rate_shift=0.0, vacancy_shift=0.0, tenant_shift=0.0):
    x = df.copy()
    x['cap_rate_expansion_stress'] = (x['cap_rate_expansion'] + cap_rate_shift).clip(0.0, 0.12)
    x['vacancy_rate_stress'] = (x['vacancy_rate'] + vacancy_shift).clip(0.0, 0.40)
    x['tenant_conc_stress'] = (x['tenant_concentration'] + tenant_shift).clip(0.10, 0.95)

    x['lgd_base_stress'] = (
        0.18
        + 0.34 * (x['lvr_at_default'] - 0.55).clip(lower=0, upper=0.35)
        + 0.20 * ((1.40 - x['dscr']) / 1.40).clip(lower=0, upper=1.00)
        + 0.08 * ((4.0 - x['wale_years']) / 4.0).clip(lower=0, upper=1.00)
        + 0.16 * x['vacancy_rate_stress']
        + 0.15 * x['tenant_conc_stress']
        + 0.10 * x['forced_sale_flag']
        - 0.05 * x['refinance_flag']
    ).clip(0.05, 0.99)

    x['downturn_scalar_stress'] = (
        1.00
        + 2.10 * x['cap_rate_expansion_stress']
        + 0.45 * x['vacancy_rate_stress']
        + 0.12 * x['forced_sale_flag']
    ).clip(1.00, 1.90)

    x['lgd_downturn_stress'] = (x['lgd_base_stress'] * x['downturn_scalar_stress']).clip(0.0, 0.99)

    overlay = (
        0.010
        + 0.020 * x['forced_sale_flag']
        + 0.012 * x['vacancy_rate_stress']
        + 0.012 * (1.0 - x['refinance_prob'])
    ).clip(0.010, 0.070)
    x['lgd_final_stress'] = (x['lgd_downturn_stress'] + overlay).clip(0.0, 0.99)

    time_to_recovery_stress = (
        x['time_to_recovery_months_base']
        * (1.10 + 0.30 * x['cap_rate_expansion_stress'] + 0.08 * x['forced_sale_flag'])
    ).clip(6.0, 72.0)

    return {
        'ead_weighted_lgd_base': exposure_weighted_average(x, 'lgd_base_stress', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(x, 'lgd_downturn_stress', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(x, 'lgd_final_stress', 'ead'),
        'mean_time_to_recovery_months': float(time_to_recovery_stress.mean()),
    }


scenarios = [
    {'scenario': 'base', 'cap_rate_shift': 0.00, 'vacancy_shift': 0.00, 'tenant_shift': 0.00},
    {'scenario': 'cap_rate_+100bps', 'cap_rate_shift': 0.01, 'vacancy_shift': 0.00, 'tenant_shift': 0.00},
    {'scenario': 'vacancy_+5pp', 'cap_rate_shift': 0.00, 'vacancy_shift': 0.05, 'tenant_shift': 0.00},
    {'scenario': 'tenant_conc_+10pp', 'cap_rate_shift': 0.00, 'vacancy_shift': 0.00, 'tenant_shift': 0.10},
    {'scenario': 'combined_stress', 'cap_rate_shift': 0.02, 'vacancy_shift': 0.07, 'tenant_shift': 0.12},
]

sens_rows = []
for s in scenarios:
    stats = run_cre_sensitivity(
        cre,
        cap_rate_shift=s['cap_rate_shift'],
        vacancy_shift=s['vacancy_shift'],
        tenant_shift=s['tenant_shift'],
    )
    sens_rows.append({
        'scenario': s['scenario'],
        'ead_weighted_lgd_base': stats['ead_weighted_lgd_base'],
        'ead_weighted_lgd_downturn': stats['ead_weighted_lgd_downturn'],
        'ead_weighted_lgd_final': stats['ead_weighted_lgd_final'],
        'mean_time_to_recovery_months': stats['mean_time_to_recovery_months'],
    })

sensitivity_df = pd.DataFrame(sens_rows)
display(sensitivity_df.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(sensitivity_df['scenario'], sensitivity_df['ead_weighted_lgd_final'], color='#c44e52')
ax.set_title('CRE Investment Sensitivity - EAD-Weighted Final LGD')
ax.set_ylabel('Final LGD')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'cre_lgd_sensitivity.png'), dpi=140, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'cre_investment_downturn_sensitivity.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)



## 6. Validation Checks and Exports

**Recovery-time metric definition (cross-product standard):**
`time_to_recovery_months_base` is the expected months from default to economic resolution (refinance settlement or forced-sale completion), including workout/marketing lag.


In [ ]:
checks = pd.DataFrame([
    {'check_name': 'lgd_base_between_zero_and_one', 'passed': cre['lgd_base'].between(0, 1).all()},
    {'check_name': 'downturn_not_below_base', 'passed': (cre['lgd_downturn'] >= cre['lgd_base']).all()},
    {'check_name': 'final_not_below_downturn', 'passed': (cre['lgd_final'] >= cre['lgd_downturn']).all()},
    {'check_name': 'final_lgd_below_one', 'passed': (cre['lgd_final'] <= 1).all()},
    {'check_name': 'recovery_time_positive', 'passed': (cre['time_to_recovery_months_base'] > 0).all()},
])
display(checks)

cre.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'cre_investment_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'cre_investment_segment_summary.csv'), index=False)
path_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'cre_investment_resolution_path_summary.csv'), index=False)
sensitivity_df['mean_recovery_time_months'] = sensitivity_df['mean_time_to_recovery_months']

sensitivity_df.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'cre_investment_sensitivity.csv'), index=False)
checks.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'cre_investment_validation_checks.csv'), index=False)

print('Saved CRE outputs to outputs/tables/:')
print('- cre_investment_loan_level_output.csv')
print('- cre_investment_segment_summary.csv')
print('- cre_investment_resolution_path_summary.csv')
print('- cre_investment_sensitivity.csv')
print('- cre_investment_validation_checks.csv')



## Assumptions and Limitations

- CRE investment outcomes use synthetic proxy data for interview-ready demonstration.
- Refinance vs forced-sale assignment is modelled via transparent proxy rules on LVR, DSCR, WALE, vacancy, and tenant concentration.
- Cap-rate expansion and vacancy stress are simplified macro proxies and should be recalibrated on internal workout history before production use.


---

## APS 113 Calibration Layer — CRE Investment

> **SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY**
>
> This section adds a full APS 113-aligned calibration loop on top of the
> proxy baseline above. Workout data is synthetically generated (2014-2024,
> 10-year window) to demonstrate methodology; real production calibration
> requires an internal workout tape.

### Calibration Pipeline (APS 113 s.32-68)

| Step | Function | APS 113 |
|------|----------|---------|
| 1 | Load/generate historical workout data | s.44, Att A |
| 2 | `compute_realised_lgd()` — LIP costs, cure leg | s.32-34 |
| 3 | `classify_economic_regime()` + `assign_regime_to_workouts()` | s.43, s.46 |
| 4 | `segment_lgd()` — product-specific segment keys | s.52 |
| 5 | `compute_long_run_lgd()` — vintage-EWA method | s.43-44 |
| 6 | `compare_model_vs_actual()` — proxy vs realised | s.60-62 |
| 7 | `apply_downturn_overlay()` + `apply_correlation_adjustment()` | s.46-50, s.55-57 |
| 8 | `MoCRegister` + `apply_moc()` — 5 APS 113 s.65 sources | s.63-65 |
| 9 | `apply_regulatory_floor()` — 25% per APS 113 s.58 | s.58 |
| 10 | Export 9 CSV outputs | — |
| 11 | `run_full_validation_suite()` — Gini, HL, PSI, OOT | s.66-68 |

**Correct APS 113 order:** LR-LGD → downturn overlay → correlation adj →
MoC → floor (MoC applied to downturn LGD, not long-run LGD, per s.63).


In [ ]:
# APS 113 Calibration Layer — imports and configuration
import os, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

from src.calibration_utils import (
    compute_realised_lgd,
    segment_lgd,
    compute_long_run_lgd,
    compare_model_vs_actual,
    classify_economic_regime,
    assign_regime_to_workouts,
    apply_downturn_overlay,
    apply_correlation_adjustment,
    build_lgd_pd_annual_series,
    estimate_lgd_pd_correlation,
    MoCRegister,
    apply_moc,
    apply_regulatory_floor,
    run_full_validation_suite,
    generate_compliance_map,
    export_compliance_map,
    CALIBRATION_STEP_ORDER,
)
from src.generators import GENERATOR_MAP, generate_all_historical_workouts

PRODUCT = "cre_investment"
SEGMENT_KEYS = ['asset_class_cre', 'lvr_band']
MODEL_LGD_COL = "lgd_final"

HISTORY_DIR = Path('..') / 'data' / 'generated' / 'historical'
OUTPUTS_DIR = Path('..') / 'outputs' / 'tables'
UPSTREAM_PARQUET = Path('..') / 'data' / 'exports' / 'macro_regime_flags.parquet'

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Product: {PRODUCT}")
print(f"Segment keys: {SEGMENT_KEYS}")
print(f"APS 113 calibration pipeline — step order:")
for step, fn, ref in CALIBRATION_STEP_ORDER:
    print(f"  Step {step:>2}: {fn:<35} {ref}")


In [ ]:
# ── STEP 1: Load or generate historical workout data (APS 113 s.44, Att A) ─
parquet_path = HISTORY_DIR / f"{PRODUCT}_workouts.parquet"

if parquet_path.exists():
    cal_loans = pd.read_parquet(parquet_path)
    # cashflows stored alongside or re-generated
    try:
        cal_cashflows = pd.read_parquet(
            HISTORY_DIR / f"{PRODUCT}_cashflows.parquet"
        )
    except FileNotFoundError:
        cal_cashflows = None
    print(f"Loaded {len(cal_loans):,} historical workout loans from Parquet")
else:
    print(f"Parquet not found — generating synthetic workout data for {PRODUCT}")
    results = generate_all_historical_workouts(
        seed=42, output_dir=HISTORY_DIR, write_parquet=True, products=[PRODUCT]
    )
    cal_loans = results[PRODUCT]["loans"]
    cal_cashflows = results[PRODUCT].get("cashflows")
    print(f"Generated {len(cal_loans):,} synthetic workout loans (2014-2024)")

print(f"Date range: {cal_loans['default_date'].min()} to {cal_loans['default_date'].max()}")
print(f"Columns: {list(cal_loans.columns)}")

# ── STEP 2: Compute realised LGD (APS 113 s.32-34) ─────────────────────────
# LIP costs (Loss Identification Period, ~90 days) auto-detected if cashflows available
lgd_df = compute_realised_lgd(
    loans=cal_loans,
    cashflows=cal_cashflows,
    ead_col="ead_at_default",
    recovery_col="recovery_amount",
    cost_col="direct_costs",
    lip_window_days=90,
)
print(f"\nStep 2: Realised LGD computed")
print(f"  EAD-weighted realised LGD: {(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum():.2%}")
display(lgd_df[['realised_lgd', 'ead_at_default']].describe().round(4))

# ── STEP 3: Economic regime classification (APS 113 s.43, s.46-50) ─────────
upstream_path = str(UPSTREAM_PARQUET) if UPSTREAM_PARQUET.exists() else None
regime_df = classify_economic_regime(
    upstream_parquet_path=upstream_path,
    method="upstream_first",
)
print(f"\nStep 3: Economic regimes classified")
print(f"  Data source: {regime_df['data_source'].iloc[0]}")
display(regime_df[['year', 'regime', 'is_downturn_period', 'data_source']].head(12))

lgd_df = assign_regime_to_workouts(lgd_df, regime_df)
downturn_pct = lgd_df.get('is_downturn_period', pd.Series([False])).mean()
print(f"  Downturn observations: {downturn_pct:.1%} of portfolio")

# ── STEP 4: Segment by product-specific keys (APS 113 s.52) ────────────────
segmented_df = segment_lgd(lgd_df, SEGMENT_KEYS)
low_count = segmented_df[segmented_df.get('segment_flag', '') == 'low_count'] if 'segment_flag' in segmented_df.columns else pd.DataFrame()
print(f"\nStep 4: Segmentation complete")
print(f"  Segments: {segmented_df.groupby(SEGMENT_KEYS, observed=True).ngroups}")
if not low_count.empty:
    print(f"  Low-count segments flagged (<20 obs): {len(low_count)}")

# ── STEP 5: Long-run LGD — vintage-EWA (APS 113 s.43-44) ─────────────────
lr_lgd_df = compute_long_run_lgd(
    segmented_df,
    segment_keys=SEGMENT_KEYS,
    method="vintage_ewa",
)
print(f"\nStep 5: Long-run LGD by segment (vintage-EWA)")
display(lr_lgd_df.round(4))
lr_lgd_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_long_run_lgd_by_segment.csv", index=False)

# ── STEP 6: Compare model vs actual (APS 113 s.60-62) ──────────────────────
# 'model_lgd' here = proxy model LGD from the section above (lgd_final if present)
if MODEL_LGD_COL in cal_loans.columns:
    compare_input = lgd_df.merge(
        cal_loans[['loan_id', MODEL_LGD_COL] if 'loan_id' in cal_loans.columns else [MODEL_LGD_COL]],
        left_index=True, right_index=True, how='left',
    ) if MODEL_LGD_COL not in lgd_df.columns else lgd_df
else:
    compare_input = lgd_df.copy()
    compare_input['model_lgd'] = lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else 0.25

model_col_to_use = MODEL_LGD_COL if MODEL_LGD_COL in compare_input.columns else 'model_lgd'
mva_df = compare_model_vs_actual(
    compare_input,
    model_lgd_col=model_col_to_use,
    segment_keys=SEGMENT_KEYS,
)
print(f"\nStep 6: Model vs actual comparison")
display(mva_df.round(4))
mva_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_model_vs_actual_comparison.csv", index=False)

# ── STEP 7: Downturn overlay + Frye-Jacobs correlation adj (s.46-50, s.55-57)
# Reuses apply_downturn_overlay from existing lgd_calculation.py
dt_lgd = apply_downturn_overlay(segmented_df, product=PRODUCT)
print(f"\nStep 7: Downturn overlay applied")
if 'lgd_downturn' in dt_lgd.columns:
    ewa_dt = (dt_lgd['lgd_downturn'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    ewa_lr = (dt_lgd['realised_lgd'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    print(f"  EWA Long-run LGD:  {ewa_lr:.2%}")
    print(f"  EWA Downturn LGD:  {ewa_dt:.2%}")
    downturn_col = 'lgd_downturn'
else:
    dt_lgd['lgd_downturn'] = dt_lgd['realised_lgd'] * 1.15  # fallback scalar
    downturn_col = 'lgd_downturn'

# Frye-Jacobs correlation adjustment (APS 113 s.55-57)
try:
    lgd_ts, pd_ts = build_lgd_pd_annual_series(dt_lgd)
    macro_for_corr = regime_df.rename(columns={'gdp_growth': 'gdp_growth_yoy'})
    corr_result = estimate_lgd_pd_correlation(lgd_ts, pd_ts, macro_for_corr)
    dt_lgd['lgd_downturn_corr_adj'] = apply_correlation_adjustment(
        dt_lgd[downturn_col], corr_result['rho'], corr_result['macro_shock_std']
    )
    downturn_col = 'lgd_downturn_corr_adj'
    print(f"  Frye-Jacobs rho={corr_result['rho']:.3f}, adj factor={corr_result['lgd_dt_adjustment_factor']:.3f}")
except Exception as exc:
    print(f"  Frye-Jacobs skipped: {exc}")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_downturn_lgd_by_segment.csv", index=False)

# ── STEP 8: MoC register + apply MoC (AFTER downturn — APS 113 s.63-65) ───
# Determine regime data source for MoC model_approximation component
data_src = regime_df['data_source'].iloc[0] if 'data_source' in regime_df.columns else 'synthetic'
n_downturn_yrs = int(regime_df['is_downturn_period'].sum()) if 'is_downturn_period' in regime_df.columns else 0

psi_approx = 0.05  # placeholder — full PSI computed in Step 11
bias_approx = float(mva_df['bias'].abs().mean()) if 'bias' in mva_df.columns else 0.02

moc_register = MoCRegister(product=PRODUCT, regime_data_source=data_src)
moc_df = moc_register.build_moc_register(
    segment_df=segmented_df,
    segment_keys=SEGMENT_KEYS,
    n_downturn_vintages=n_downturn_yrs,
    psi_value=psi_approx,
    backtesting_bias=bias_approx,
)
print(f"\nStep 8: MoC register built")
display(moc_df.round(4))
moc_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_moc_register.csv", index=False)

lgd_with_moc = apply_moc(dt_lgd[downturn_col], moc_df, segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None)
dt_lgd['lgd_with_moc'] = lgd_with_moc
moc_ewa = (lgd_with_moc * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
print(f"  EWA LGD after MoC: {moc_ewa:.2%}")

# ── STEP 9: Regulatory floors (APS 113 s.58) ──────────────────────────────
dt_lgd['lgd_final_calibrated'] = apply_regulatory_floor(dt_lgd['lgd_with_moc'], product=PRODUCT)
final_ewa = (dt_lgd['lgd_final_calibrated'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
floor_binding_pct = (dt_lgd['lgd_final_calibrated'] > dt_lgd['lgd_with_moc']).mean()
print(f"\nStep 9: Regulatory floor applied")
print(f"  EWA Final Calibrated LGD: {final_ewa:.2%}")
print(f"  Floor binding for: {floor_binding_pct:.1%} of loans")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_final_calibrated_lgd.csv", index=False)

# ── STEP 10: Export all outputs ────────────────────────────────────────────
# Already exported: long_run_lgd_by_segment, model_vs_actual, downturn_lgd, moc_register, final_calibrated_lgd
# Export remaining:
lgd_df[['realised_lgd', 'ead_at_default']].assign(product=PRODUCT).to_csv(
    OUTPUTS_DIR / f"{PRODUCT}_historical_workouts.csv", index=False
)
regime_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_regime_classification.csv", index=False)

# Calibration adjustments summary
cal_adj_summary = pd.DataFrame({
    'product': [PRODUCT],
    'ewa_realised_lgd': [(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum()],
    'ewa_long_run_lgd': [lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None],
    'ewa_downturn_lgd': [(dt_lgd.get('lgd_downturn', dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_with_moc': [(dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_final': [final_ewa],
    'floor_binding_pct': [floor_binding_pct],
    'regime_data_source': [data_src],
    'n_loans': [len(lgd_df)],
    'calibration_date': [pd.Timestamp.today().date()],
})
cal_adj_summary.to_csv(OUTPUTS_DIR / f"{PRODUCT}_calibration_adjustments.csv", index=False)
print(f"\nStep 10: All outputs exported to {OUTPUTS_DIR}")

# ── STEP 11: Full validation suite (APS 113 s.66-68) ──────────────────────
print(f"\nStep 11: Running full validation suite...")
try:
    val_results = run_full_validation_suite(
        loans=dt_lgd,
        predicted_col='lgd_final_calibrated',
        actual_col='realised_lgd',
        segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None,
        date_col='default_date' if 'default_date' in dt_lgd.columns else None,
        product=PRODUCT,
    )
    print(f"  Gini: {val_results.get('gini', 'n/a')}")
    print(f"  Calibration ratio: {val_results.get('calibration_ratio', 'n/a')}")
    print(f"  PSI: {val_results.get('psi', 'n/a')}")
    if 'summary_table' in val_results:
        val_results['summary_table'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_validation_report.csv", index=False
        )
    if 'backtest_results' in val_results:
        val_results['backtest_results'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_backtest_results.csv", index=False
        )
except Exception as exc:
    print(f"  Validation suite error (non-fatal): {exc}")


In [ ]:
# ── Calibration summary waterfall ──────────────────────────────────────────
import matplotlib.pyplot as plt

try:
    stages = {
        'Realised LGD\n(2014-2024)': (lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum(),
        'Long-run LGD\n(vintage-EWA)': lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None,
        'Downturn LGD': (dt_lgd.get(downturn_col, dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        '+ MoC': (dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        'Final\n(Floor Applied)': final_ewa,
    }
    labels = [k for k, v in stages.items() if v is not None]
    values = [v for v in stages.values() if v is not None]
    colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad'][:len(labels)]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_ylabel('EAD-Weighted LGD')
    ax.set_title(f'APS 113 Calibration Waterfall — CRE Investment')
    ax.set_ylim(0, max(values) * 1.35)
    ax.axhline(values[-1], color='black', ls=':', lw=1, label=f'Final: {values[-1]:.1%}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        Path('..') / 'outputs' / 'figures' / f'cre_investment_calibration_waterfall.png',
        dpi=150, bbox_inches='tight',
    )
    plt.show()
except Exception as exc:
    print(f"Waterfall chart error (non-fatal): {exc}")

# ── APS 113 compliance snapshot ─────────────────────────────────────────────
compliance_df = generate_compliance_map(
    calibration_results={PRODUCT: {'long_run_lgd_by_segment': True, 'calibration_steps': True}},
    moc_registers={PRODUCT: moc_df},
    regime_data_source=data_src,
    products=[PRODUCT],
)
print("\n=== APS 113 Compliance Snapshot ===")
display(compliance_df[['section_ref', 'requirement', 'status', 'reviewer_note']].set_index('section_ref'))
export_compliance_map(compliance_df, OUTPUTS_DIR / f"cre_investment_aps113_compliance.csv")

# Final summary
print("\n=== Calibration Summary ===")
display(cal_adj_summary.round(4))
print(f"\nAll calibration outputs in: {OUTPUTS_DIR}")
print(f"SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY")
